In [ ]:
import sys
sys.path.append('../../utils')
import utility
import math
import itertools
import random
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
from openai.lib._parsing._completions import type_to_response_format_param
import json
import re
from z3 import *

from typing import Literal

# load_dotenv()
# assert os.getenv("OPENAI_API_KEY") is not None, "You must set your OpenAI API key in the .env file"

# openai_client = OpenAI()
seeds_considered = [3, 12, 107, 85, 26]

In [ ]:
folio_dict = utility.make_jsonl_into_list_dictionaries('../../datasets/folio-train-clean.jsonl')
NL_sentences = {}
FOL_formulas = {}
story_id_already_seen = set()
for item in folio_dict:
    story_id = item['story_id']
    if story_id not in story_id_already_seen:
        story_id_already_seen.add(story_id)
        NL_sentences[story_id] = item['premises']
        FOL_formulas[story_id] = item['premises-FOL']
# remove the story 236 because logically problematic (PValue doesn't mantain the same arity in the FOL translations)
NL_sentences.pop(236)
FOL_formulas.pop(236)


        
# Delete the sentences in the stories that have the Xor symbol ⊕ (it is treated inconsistently in the dataset)
for i in range(2):
    for story_id in FOL_formulas.keys():
        for sentence in FOL_formulas[story_id]:
            if '⊕' in sentence:
                index = FOL_formulas[story_id].index(sentence)
                FOL_formulas[story_id].pop(index)
                NL_sentences[story_id].pop(index)


In [ ]:
constant_meaning = utility.load_pickle('../../perturbations and translations/Constant_meaning_FOLIO.pkl')
relational_meaning = utility.load_pickle('../../perturbations and translations/Relational_meaning_FOLIO.pkl')

system_prompt = """
You are an expert evaluator specializing in translating natural language sentences into a logical formalism. Your task is to formalize a given sentence in First Order Logic (FOL).

Instructions:
You can use the following symbols:
- logical symbols: ∀ (for all), ∃ (exists), → (implies), ↔ (is equivalent to), ∧ (and), ∨ (or), ¬ (not);
- predicate symbols: <predicate_symbols>
- contant symbols: <constant_symbols>
- variable symbols: x,y,z,...
- non logical symbols: parenthesis ()


Output Format:
Return the First Order Logic sentence that best represents the meaning of the given sentence.

Input Format:
Sentence: <sentence>
"""

user_prompt = """
Sentence: <sentence>
"""


class response_format(BaseModel):
    reasoning: str
    answer: str

system_prompts = {}
user_prompts = {}
var = ['x', 'y', 'z', 'w', 'v']

for story_id in NL_sentences.keys():
    for i, sentence in enumerate(NL_sentences[story_id]):
        key_ref = f'{story_id}_{i}'
        sp = system_prompt

        pred_symbols = ''
        dict_rm = relational_meaning[story_id]['dict_pos']
        t = {}
        for key, item in dict_rm.items():
            i = 0
            while '{}' in item:
                count = item.index('{}')
                item = item[:count] + var[i] + item[count+2:] 
                i += 1
            new_key = key + '('
            for j in range(i):
                new_key = new_key + var[j] + ','
            new_key = new_key[:-1] + ')'
            
            pred_symbols = pred_symbols + new_key + ': ' + '(' + item + ')' '\n'
        
        const_symbols = ''
        for key, item in constant_meaning[story_id].items():
            const_symbols = const_symbols + key + ': ' + '(' + item + ')' +'\n'
        sp = sp.replace('<predicate_symbols>', pred_symbols)
        sp = sp.replace('<constant_symbols>', const_symbols)

        system_prompts[key_ref] = sp
        user_prompts[key_ref] = user_prompt.replace('<sentence>', sentence)


In [ ]:
# Call API
model = 'gpt-4o-mini'
list_requests = []
for seed in seeds_considered:
    custom_id = f'FOLIO_log_transl_{seed}'
    requests = utility.create_openai_list_requests_for_batch_completions(model, custom_id, system_prompts, user_prompts, type_to_response_format_param(response_format), seed = seed, max_tokens = 2500)
    list_requests += requests


# Save the requests in a json file and call openai 
utility.make_list_dictionaries_into_jsonl(list_requests, f'Requests/FOLIO_log_transl_{model}_seed.jsonl')

# Call API (Remove the comments in case you want to call the API)
# FOLIO_log_transl_4omini_seed, FOLIO_log_transl_4omini_batch_seed = utility.call_openai_api_batch_completions(list_requests, 'Requests', f'FOLIO_log_transl_{model}_seed.jsonl')

# save the batch id
# utility.save_pickle(FOLIO_log_transl_4omini_seed, f'Requests/batch_ids/FOLIO_log_transl_{model}_seed.pkl')


In [ ]:
# load the batch
model = 'gpt-4o-mini'
FOLIO_log_transl_4omini_id_seed = utility.load_pickle(f'Requests/batch_ids/FOLIO_log_transl_{model}_seed.pkl')
FOLIO_log_transl_4omini_batch_seed = openai_client.batches.retrieve(FOLIO_log_transl_4omini_id_seed)
print(FOLIO_log_transl_4omini_batch_seed.status)
print(FOLIO_log_transl_4omini_batch_seed)

if utility.extract_batch_errors_openai(FOLIO_log_transl_4omini_batch_seed):
    print('Error file is present.')
else:
    output = utility.extract_batch_outputs_openai(FOLIO_log_transl_4omini_batch_seed)
    if not output:
        print('No output file available')

utility.save_pickle(output, f'Results/FOLIO_log_transl_{model}_seed.pkl')

In [ ]:
# Clean the results and save them in order to get the clean results and then the performances
model = 'gpt-4o-mini'
output = utility.load_pickle(f'Results/FOLIO_log_transl_{model}_seed.pkl')

list_error_parsing = {}
list_responses = {}


for seed in seeds_considered:
    list_responses[seed] = []
    list_error_parsing[seed] = []


for item in output:
    seed = int(item['custom_id'].split('_')[-3])
    number_instance = int(item['custom_id'].split('_')[-1]) 
    story_id = int(item['custom_id'].split('_')[-2])
    try: 
        message = json.loads(item['response']['body']['choices'][0]['message']['content'])
        answer = str(message['answer'])
        reasoning = message['reasoning']
        list_responses[seed].append({
                'story_id' : story_id,
                'number_instance' : number_instance,
                'answer': answer,
                'reasoning': reasoning
            })
    except:
        list_error_parsing[seed].append(f'{story_id}_{number_instance}')

for seed in seeds_considered:
    print('Seed :', seed)
    print('list_error_parsing', list_error_parsing[seed])
    print('length list error parsing', len(list_error_parsing[seed]))
    print('_'*20)

utility.save_pickle(list_responses, f'Results/Clean Results/FOLIO_log_transl_{model}_seed.pkl')

print(list_responses)


In [ ]:
# Translate in the new formalism (rename of predicate and constant symbols) the ground truths and the predictions
model = 'gpt-4o-mini'
clean_output = utility.load_pickle(f'Results/Clean Results/FOLIO_log_transl_{model}_seed.pkl')
print(clean_output)


predictions = {}
FOL_formulas_for_Z3 = {}
predictions_for_Z3 = {}
list_exception = {}
list_exception_use_equality = {}

for seed in seeds_considered:
    predictions[seed] = {}
    FOL_formulas_for_Z3[seed] = {}
    predictions_for_Z3[seed] = {}
    list_exception[seed] = []
    list_exception_use_equality[seed] = []


for seed in seeds_considered:
    for story_id in NL_sentences.keys():
        predictions[seed][story_id] = ['' for i in NL_sentences[story_id]]
        FOL_formulas_for_Z3[seed][story_id] = ['' for i in NL_sentences[story_id]]
        predictions_for_Z3[seed][story_id] = ['' for i in NL_sentences[story_id]]
    for line in clean_output[seed]:
        story_id, number_instance, answer = line['story_id'], line['number_instance'], line['answer']
        predictions[seed][story_id][number_instance] = answer
        

    for story_id in FOL_formulas.keys():
        for i, formula in enumerate(FOL_formulas[story_id]):
            try:
                
                x = formula
                y = predictions[seed][story_id][i]
                # - put in a nicer form to be parsed:
                # replace [ or ] with ( or )
                y = y.replace('[', '(')
                y = y.replace(']', ')')
                # - put a space after the quantifiers and remove the space between the quantifier and the varibale
                y = y.replace('∀ ', '∀')
                y = y.replace('∃ ', '∃')
                # - add a space after the quantified variable
                pattern = r"∀([a-zA-Z]+)\("
                replacement = r"∀\1 ("
                y = re.sub(pattern, replacement, y)

                pattern = r"∃([a-zA-Z]+)\("
                replacement = r"∃\1 ("
                y = re.sub(pattern, replacement, y)
                # remove final dots or comma for ending 
                if y[-1] == ',' or y[-1] == '.':
                    y = y[:-1]
                
                # Order the constant, and predicate symbols from the longer to the shorter (in this way we don't have problem if we have two constants like 'son' and 'jamson')
                r, c1, v1 = utility.extract_rel_const_var(x)
                r2, c2, v2 = utility.extract_rel_const_var(y)
                c = c1.union(c2)
                v = v1.union(v2)
                r.update(r2)


                list_constants = sorted(list(c), key=len, reverse=True)

                unary_symbols = [rel for rel in r.keys() if r[rel] == 1]
                binary_symbols = [rel for rel in r.keys() if r[rel] == 2]
                ternary_symbols = [rel for rel in r.keys() if r[rel] == 3]
                
                
                list_replacements = sorted(list_constants + unary_symbols + binary_symbols + ternary_symbols, key=len, reverse=True)
            
                for item in list_replacements:
                    if item in list_constants:
                        index = list_constants.index(item) + 1
                        x = x.replace(item, 'A' + str(index))
                        y = y.replace(item, 'A' + str(index))
                    if item in unary_symbols:
                        index = unary_symbols.index(item) + 1
                        x = x.replace(item, 'U'+str(index))
                        y = y.replace(item, 'U'+str(index))
                    if item in binary_symbols:
                        index = binary_symbols.index(item) + 1
                        x = x.replace(item, 'B'+str(index)) 
                        y = y.replace(item, 'B'+str(index))
                    if item in ternary_symbols:
                        index = ternary_symbols.index(item) + 1
                        x = x.replace(item, 'T'+str(index))
                        y = y.replace(item, 'T'+str(index))
                

                FOL_formulas_for_Z3[seed][story_id][i] = utility.parse_formula_SMTLIB_universal(x)
                predictions_for_Z3[seed][story_id][i] = utility.parse_formula_SMTLIB_universal(y)
                    
            except:
                list_exception[seed].append(f'{story_id}_{i}')
                if ('=' in predictions[seed][story_id][i]) or ('≠' in predictions[seed][story_id][i]):
                    list_exception_use_equality[seed].append(f'{story_id}_{i}')

    print('Seed:', seed)
    print('list_exception = ', list_exception[seed])
    print('length list_exception = ', len(list_exception[seed]))
    print('instances in which equality is used', list_exception_use_equality[seed])
    print('length list_exception with equality = ', len(list_exception_use_equality[seed]))
    print('_'*20)

In [ ]:
# Remove the item that are in list_ep
model = 'gpt-4o-mini'

correct_translations = {}
list_correct_positions = {}
list_incorrect_positions = {}
list_done = {}

for seed in seeds_considered:
    correct_translations[seed] = {}
    list_correct_positions[seed] = {}
    list_incorrect_positions[seed] = {}
    list_done[seed] = {}

for seed in seeds_considered:
    for story_id in NL_sentences.keys():
        correct_translations[seed][story_id] = ['' for i in NL_sentences[story_id]]
        list_correct_positions[seed][story_id] = []
        list_incorrect_positions[seed][story_id] = []
        list_done[seed][story_id] = []
        for i, sent in enumerate(NL_sentences[story_id]):
            key = f'{story_id}_{i}'
            if key in list_exception[seed]:
                if key in list_exception_use_equality[seed]:
                    correct_translations[seed][story_id][i] = 'equality_used'
                else:
                    correct_translations[seed][story_id][i] = 'parsing_error'
            else:
                pred = predictions_for_Z3[seed][story_id][i]
                correct_translations[seed][story_id][i] = utility.check_equivalence_with_parsed_formulas(pred, FOL_formulas_for_Z3[seed][story_id][i])
            
                if correct_translations[seed][story_id][i] == True:
                    list_correct_positions[seed][story_id].append(i)
                elif correct_translations[seed][story_id][i] == False:
                    list_incorrect_positions[seed][story_id].append(i)
                
            list_done[seed][story_id].append(i)

    # Now let's check if the translations are correct
    count_equivalent = 0
    count_not_equivalent = 0
    count_parsing_error = 0
    count_solver_failed = 0
    count_equality_used = 0
    count_total = 0
    for story_id in correct_translations[seed].keys():
        for item in correct_translations[seed][story_id]:
            count_total += 1
            if item == True:
                count_equivalent += 1
            elif item == False:
                count_not_equivalent += 1
            elif item == 'Solver_failed':
                count_solver_failed += 1
            elif item == 'parsing_error':
                count_parsing_error += 1
            elif item == 'equality_used':
                count_equality_used += 1

    print('Model: ', model)
    print('Seed: ', seed)
    print('No. of instances processed in total: ', count_total)        
    print('Solver failed', count_solver_failed, 'times')
    print('Parsing errors: ', count_parsing_error)
    print('Equality used', count_equality_used, 'times')
    print('No. of times an equivalent formula is output: ', count_equivalent)
    print('Accuracy: ', count_equivalent/count_total)
    print(list_incorrect_positions)

utility.save_pickle(correct_translations, f'')

# Do the same for o3-mini

In [ ]:
# Call API
model = 'o3-mini'
seeds_considered = [3, 12, 107, 85, 26]

list_requests = []
for seed in seeds_considered:
    custom_id = f'FOLIO_log_transl_{seed}'
    requests = utility.create_openai_list_requests_for_batch_completions(model, custom_id, system_prompts, user_prompts, type_to_response_format_param(response_format), seed = seed, max_tokens = 10000)
    list_requests += requests

# Save the requests in a json file and call openai 
utility.make_list_dictionaries_into_jsonl(list_requests, f'Requests/FOLIO_log_transl_{model}_seed.jsonl')

# Call API (Remove the comments from above if you want to call the APIs)
#FOLIO_log_transl_o3mini_seed, FOLIO_log_transl_o3mini_batch_seed = utility.call_openai_api_batch_completions(list_requests, 'Requests', f'FOLIO_log_transl_{model}_seed.jsonl')

# save the batch id
# utility.save_pickle(FOLIO_log_transl_o3mini_seed, f'Requests/batch_ids/FOLIO_log_transl_{model}_seed.pkl')



In [ ]:
# load the batch
model = 'o3-mini'
FOLIO_log_transl_o3mini_id_seed = utility.load_pickle(f'Requests/batch_ids/FOLIO_log_transl_{model}_seed.pkl')
FOLIO_log_transl_o3mini_batch_seed = openai_client.batches.retrieve(FOLIO_log_transl_o3mini_id_seed)
print(FOLIO_log_transl_o3mini_batch_seed.status)
print(FOLIO_log_transl_o3mini_batch_seed)

if utility.extract_batch_errors_openai(FOLIO_log_transl_o3mini_batch_seed):
    print('Error file is present.')
else:
    output = utility.extract_batch_outputs_openai(FOLIO_log_transl_o3mini_batch_seed)
    if not output:
        print('No output file available')

utility.save_pickle(output, f'Results/FOLIO_log_transl_{model}_seed.pkl')

In [ ]:
# Clean the results and save them in order to get the clean results and then the performances
model = 'o3-mini'
output = utility.load_pickle(f'Results/FOLIO_log_transl_{model}_seed.pkl')

list_error_parsing = {}
list_responses = {}

for seed in seeds_considered:
    list_responses[seed] = []
    list_error_parsing[seed] = []

for item in output:
    seed = int(item['custom_id'].split('_')[-3])
    number_instance = int(item['custom_id'].split('_')[-1]) 
    story_id = int(item['custom_id'].split('_')[-2])
    try: 
        message = json.loads(item['response']['body']['choices'][0]['message']['content'])
        answer = str(message['answer'])
        reasoning = message['reasoning']
        list_responses[seed].append({
                'story_id' : story_id,
                'number_instance' : number_instance,
                'answer': answer,
                'reasoning': reasoning
            })
    except:
        list_error_parsing[seed].append(f'{story_id}_{number_instance}')

for seed in seeds_considered:
    print('Seed :', seed)
    print('list_error_parsing', list_error_parsing[seed])
    print('length list error parsing', len(list_error_parsing[seed]))
    print('_'*20)


utility.save_pickle(list_responses, f'Results/Clean Results/FOLIO_log_transl_{model}_seed.pkl')


In [ ]:
# Translate in the new formalism (rename of predicate and constant symbols) the ground truths and the predictions
model = 'o3-mini'
clean_output = utility.load_pickle(f'Results/Clean Results/FOLIO_log_transl_{model}_seed.pkl')


predictions = {}
FOL_formulas_for_Z3 = {}
predictions_for_Z3 = {}
list_exception = {}
list_exception_use_equality = {}

for seed in seeds_considered:
    predictions[seed] = {}
    FOL_formulas_for_Z3[seed] = {}
    predictions_for_Z3[seed] = {}
    list_exception[seed] = []
    list_exception_use_equality[seed] = []


for seed in seeds_considered:
    for story_id in NL_sentences.keys():
        predictions[seed][story_id] = ['' for i in NL_sentences[story_id]]
        FOL_formulas_for_Z3[seed][story_id] = ['' for i in NL_sentences[story_id]]
        predictions_for_Z3[seed][story_id] = ['' for i in NL_sentences[story_id]]
    for line in clean_output[seed]:
        story_id, number_instance, answer = int(line['story_id']), int(line['number_instance']), line['answer']
        predictions[seed][story_id][number_instance] = answer


    for story_id in FOL_formulas.keys():
        for i, formula in enumerate(FOL_formulas[story_id]):
            try:
                
                x = formula
                y = predictions[seed][story_id][i]

                # - put in a nicer form to be parsed:
                # replace [ or ] with ( or )
                y = y.replace('[', '(')
                y = y.replace(']', ')')
                # - put a space after the quantifiers and remove the space between the quantifier and the varibale
                y = y.replace('∀ ', '∀')
                y = y.replace('∃ ', '∃')
                # - add a space after the quantified variable
                pattern = r"∀([a-zA-Z]+)\("
                replacement = r"∀\1 ("
                y = re.sub(pattern, replacement, y)

                pattern = r"∃([a-zA-Z]+)\("
                replacement = r"∃\1 ("
                y = re.sub(pattern, replacement, y)
                # remove final dots or comma for ending 
                if y[-1] == ',' or y[-1] == '.':
                    y = y[:-1]
                

                
                # Order the constant, and predicate symbols from the longer to the shorter (in this way we don't have problem if we have two constants like 'son' and 'jamson')
                r, c1, v1 = utility.extract_rel_const_var(x)
                r2, c2, v2 = utility.extract_rel_const_var(y)
                c = c1.union(c2)
                v = v1.union(v2)
                r.update(r2)


                list_constants = sorted(list(c), key=len, reverse=True)

                unary_symbols = [rel for rel in r.keys() if r[rel] == 1]
                binary_symbols = [rel for rel in r.keys() if r[rel] == 2]
                ternary_symbols = [rel for rel in r.keys() if r[rel] == 3]
                
                
                list_replacements = sorted(list_constants + unary_symbols + binary_symbols + ternary_symbols, key=len, reverse=True)
            
                # Build a lookup dict of replacements
                replacements = {}
                for item in list_replacements:
                    if item in list_constants:
                        index = list_constants.index(item) + 1
                        replacements[item] = 'A' + str(index)
                    elif item in unary_symbols:
                        index = unary_symbols.index(item) + 1
                        replacements[item] = 'U' + str(index)
                    elif item in binary_symbols:
                        index = binary_symbols.index(item) + 1
                        replacements[item] = 'B' + str(index)
                    elif item in ternary_symbols:
                        index = ternary_symbols.index(item) + 1
                        replacements[item] = 'T' + str(index)

                # Compile one regex that matches any key
                pattern = re.compile('|'.join(map(re.escape, replacements.keys())))

                # Replace using the mapping
                def replace_all(match):
                    return replacements[match.group(0)]

                x_new = pattern.sub(replace_all, x)
                y_new = pattern.sub(replace_all, y)
                
                FOL_formulas_for_Z3[seed][story_id][i] = utility.parse_formula_SMTLIB_universal(x_new)
                predictions_for_Z3[seed][story_id][i] = utility.parse_formula_SMTLIB_universal(y_new)
            except:
                list_exception[seed].append(f'{story_id}_{i}')
                if ('=' in predictions[seed][story_id][i]) or ('≠' in predictions[seed][story_id][i]):
                    list_exception_use_equality[seed].append(f'{story_id}_{i}')

    print('Seed:', seed)
    print('list_exception_without_eq = ', [i for i in list_exception[seed] if i not in list_exception_use_equality[seed]])
    print('length list_exception_without_eq = ', len([i for i in list_exception[seed] if i not in list_exception_use_equality[seed]]))
    print('instances in which equality is used', list_exception_use_equality[seed])
    print('length list_exception with equality = ', len(list_exception_use_equality[seed]))
    print('_'*20)

In [ ]:
# Remove the item that are in list_ep
model = 'o3-mini'

correct_translations = {}
list_correct_positions = {}
list_incorrect_positions = {}
list_done = {}

for seed in seeds_considered:
    correct_translations[seed] = {}
    list_correct_positions[seed] = {}
    list_incorrect_positions[seed] = {}
    list_done[seed] = {}

for seed in seeds_considered:
    for story_id in NL_sentences.keys():
        correct_translations[seed][story_id] = ['' for i in NL_sentences[story_id]]
        list_correct_positions[seed][story_id] = []
        list_incorrect_positions[seed][story_id] = []
        list_done[seed][story_id] = []
        for i, sent in enumerate(NL_sentences[story_id]):
            key = f'{story_id}_{i}'
            if key in list_exception[seed]:
                if key in list_exception_use_equality[seed]:
                    correct_translations[seed][story_id][i] = 'equality_used'
                else:
                    correct_translations[seed][story_id][i] = 'parsing_error'
            else:
                pred = predictions_for_Z3[seed][story_id][i]
                correct_translations[seed][story_id][i] = utility.check_equivalence_with_parsed_formulas(pred, FOL_formulas_for_Z3[seed][story_id][i])
            
                if correct_translations[seed][story_id][i] == True:
                    list_correct_positions[seed][story_id].append(i)
                elif correct_translations[seed][story_id][i] == False:
                    list_incorrect_positions[seed][story_id].append(i)
                
            list_done[seed][story_id].append(i)

    # Now let's check if the translations are correct
    count_equivalent = 0
    count_not_equivalent = 0
    count_parsing_error = 0
    count_solver_failed = 0
    count_equality_used = 0
    count_total = 0
    for story_id in correct_translations[seed].keys():
        for item in correct_translations[seed][story_id]:
            count_total += 1
            if item == True:
                count_equivalent += 1
            elif item == False:
                count_not_equivalent += 1
            elif item == 'Solver_failed':
                count_solver_failed += 1
            elif item == 'parsing_error':
                count_parsing_error += 1
            elif item == 'equality_used':
                count_equality_used += 1

    print('Model: ', model)
    print('Seed: ', seed)
    print('No. of instances processed in total: ', count_total)        
    print('Solver failed', count_solver_failed, 'times')
    print('Parsing errors: ', count_parsing_error)
    print('Equality used', count_equality_used, 'times')
    print('No. of times an equivalent formula is output: ', count_equivalent)
    print('Accuracy: ', count_equivalent/count_total)
    print(list_incorrect_positions)